# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tahir-MD/FlyRank-Week-01/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
REASON_CODES = {
    "DECLINING_TREND": "Traffic trend is down over the last 30 days vs prior 30 days",
    "LOW_CTR_FOR_POSITION": "CTR is below the median for its position tier",
    "STALE_CONTENT": "Not updated in a long time relative to its age tier",
    "HIGH_VALUE_OPPORTUNITY": "High search volume/impressions but low clicks",
}
for code, desc in REASON_CODES.items():
    print(f"{code}: {desc}")

DECLINING_TREND: Traffic trend is down over the last 30 days vs prior 30 days
LOW_CTR_FOR_POSITION: CTR is below the median for its position tier
STALE_CONTENT: Not updated in a long time relative to its age tier
HIGH_VALUE_OPPORTUNITY: High search volume/impressions but low clicks


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

# ✅ Use RAW GitHub URL
df = pd.read_csv("https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv")

median_ctr_by_tier = df.groupby("position_tier")["ctr"].transform("median")

def score_row(row, median_ctr):
    score = 0
    reasons = []

    if row["trend_direction"] == "down":
        score += 2
        reasons.append("DECLINING_TREND")

    if row["ctr"] < median_ctr:
        score += 1
        reasons.append("LOW_CTR_FOR_POSITION")

    if row["days_since_last_update"] > 180:
        score += 1
        reasons.append("STALE_CONTENT")

    if row["search_volume"] > df["search_volume"].median() and row["clicks_last_30d"] < df["clicks_last_30d"].median():
        score += 1
        reasons.append("HIGH_VALUE_OPPORTUNITY")

    return score, ",".join(reasons) if reasons else "NONE"

results = df.apply(lambda r: score_row(r, median_ctr_by_tier[r.name]), axis=1)

df["action_score"] = results.apply(lambda x: x[0])
df["reason_codes"] = results.apply(lambda x: x[1])

ranked = df.sort_values("action_score", ascending=False)

import os
os.makedirs("work/outputs", exist_ok=True)

ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(ranked[["content_id", "action_score", "reason_codes"]].head(20))

                 content_id  action_score  \
3651   content_fd16e3475c29             4   
29906  content_07ce98c6085a             4   
8860   content_adfc46f3f033             4   
24454  content_7d58c3076247             4   
15089  content_149c671a2a91             4   
11489  content_5feee3994adb             4   
26242  content_55a5b1c46474             4   
22856  content_460b11dcac6a             4   
18841  content_94991fe6268c             4   
1227   content_4f241bad48a3             4   
17510  content_b694314765e5             4   
505    content_bfa3d6688324             4   
8675   content_dd2e06be1af4             4   
3507   content_074ba6ead17b             4   
20189  content_277fa742f704             4   
21984  content_02b0d6e30129             4   
22860  content_ab18b5811c02             4   
28250  content_8c66ab2089c0             4   
24557  content_84d12054c0c0             4   
6421   content_fc8cb7532683             4   

                                            reason_cod

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top20 = ranked.head(20)[[
    "content_id", "action_score", "reason_codes", "trend_direction",
    "ctr", "avg_position", "days_since_last_update", "search_volume"
]]
print(top20.to_string(index=False))

          content_id  action_score                                       reason_codes trend_direction  ctr  avg_position  days_since_last_update  search_volume
content_fd16e3475c29             4 DECLINING_TREND,LOW_CTR_FOR_POSITION,STALE_CONTENT            down 0.00           9.0                     183            0.0
content_07ce98c6085a             4 DECLINING_TREND,LOW_CTR_FOR_POSITION,STALE_CONTENT            down 0.00           5.3                     304            NaN
content_adfc46f3f033             4 DECLINING_TREND,LOW_CTR_FOR_POSITION,STALE_CONTENT            down 0.00          23.7                     183            0.0
content_7d58c3076247             4 DECLINING_TREND,LOW_CTR_FOR_POSITION,STALE_CONTENT            down 0.00           7.8                     183            0.0
content_149c671a2a91             4 DECLINING_TREND,LOW_CTR_FOR_POSITION,STALE_CONTENT            down 0.00          19.3                     235           10.0
content_5feee3994adb             4 DECLI

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# sanity check: does the score correlate suspiciously with columns that shouldn't drive it?
suspect_cols = ["client_id", "provider_used", "model_used"]  # things that shouldn't matter
for col in suspect_cols:
    print(df.groupby(col)["action_score"].mean())

# confirm no future-window columns snuck in unexamined
print("Columns used in scoring: trend_direction, ctr, days_since_last_update, search_volume, clicks_last_30d")

client_id
client_02d20bbd7e    1.500000
client_0b918943df    1.742857
client_19581e27de    1.451484
client_1a6562590e    1.000000
client_25fc0e7096    0.000000
client_2c624232cd    0.773498
client_349c41201b    1.579292
client_3fdba35f04    2.110719
client_434c9b5ae5    1.770115
client_4e07408562    1.292502
client_4ec9599fc2    1.839928
client_4fc82b26ae    1.718750
client_6208ef0f77    1.564249
client_624b60c58c    1.890208
client_7f2253d7e2    2.181208
client_8527a891e2    1.592127
client_8722616204    2.178201
client_8b940be7fb    0.642857
client_9400f1b21c    2.145570
client_98a3ab7c34    2.076271
client_9f14025af0    0.643077
client_a88a7902cb    1.851409
client_b4944c6ff0    0.889764
client_bbb965ab0c    1.421782
client_bdd2d3af3a    2.022727
client_d029fa3a95    1.975840
client_d4735e3a26    0.759494
client_d59eced1de    1.604651
client_e29c9c180c    0.519774
client_e629fa6598    1.554167
client_f369cb89fc    1.722717
client_f74efabef1    1.534433
Name: action_score, dtype: flo

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.